# Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import joblib

# Read the dataset

In [2]:
df = pd.read_csv("Loan_default_cleaned.csv")

print(df.head())
print(df.shape)
print(df.columns)

       loanid       age    income  loanamount  creditscore  monthsemployed  \
0  I38PQUQS96  0.833990  0.089693   -1.086833    -0.341492        0.590533   
1  HPSK72WA7R  1.701221 -0.823021   -0.044309    -0.731666       -1.285731   
2  C1OZ6DPJ8Y  0.166888  0.043854    0.022715    -0.775718       -0.968209   
3  V2KKSFM3UN -0.767053 -1.303452   -1.168538     1.061875       -1.718715   
4  EY08JDHTZP  1.100830 -1.592855   -1.671921     0.369631       -1.487790   

   numcreditlines  interestrate  loanterm  dtiratio  education  \
0        1.341937      0.261771 -0.001526 -0.260753          0   
1       -1.343791     -1.308350  1.412793  0.778585          2   
2        0.446694      1.156831 -0.708685 -0.823728          2   
3        0.446694     -0.967805 -0.708685 -1.170174          1   
4        1.341937     -1.052188  0.705634  0.995114          0   

   employmenttype  maritalstatus  hasmortgage  hasdependents  loanpurpose  \
0               0              0            1            

# Separate Input and Output

In [3]:
x = df.drop(columns = ["loanid", "default"])
y = df["default"]

In [4]:
print("X shape:", x.shape)
print("y shape:", y.shape)

X shape: (255347, 16)
y shape: (255347,)


# Split the dataset

In [5]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
)

# Cross-Validation on the Base Model

In [6]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Using a subset of the training data for faster CV evaluation
# We take 20% of the training data just to get a quick estimate
x_train_sample = x_train.sample(frac=0.2, random_state=42)
y_train_sample = y_train.loc[x_train_sample.index]

base_rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    n_jobs=-1, # Use all available cores
    random_state=42
)

print("Running 3-fold cross-validation on a 20% sample of training data...")
cv_scores = cross_val_score(base_rf, x_train_sample, y_train_sample, cv=3, scoring='accuracy', n_jobs=-1)

print("Cross-Validation Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


Running 3-fold cross-validation on a 20% sample of training data...
Cross-Validation Accuracy Scores: [0.88457302 0.88449111 0.88405052]
Mean CV Accuracy: 0.8843715530175018


# Hyperparameter Tuning with RandomizedSearchCV

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define a smaller, targeted parameter grid
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5]
}

# Added class_weight='balanced' to handle the severe class imbalance technically
rf_for_tuning = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')

# Changed scoring to 'f1' to optimize for Class 1 instead of overall accuracy
random_search = RandomizedSearchCV(
    estimator=rf_for_tuning,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

print("Starting RandomizedSearchCV on 20% sample...")
random_search.fit(x_train_sample, y_train_sample)

print("Best Parameters found:", random_search.best_params_)
print("Best Cross-Validated F1 Score:", random_search.best_score_)


Starting RandomizedSearchCV on 20% sample...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters found: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': None}
Best Cross-Validated F1 Score: 0.3489087216147421


# Train Final Model with Best Parameters

In [8]:
# Extract the best model from the search
best_rf = random_search.best_estimator_

# Fit the best model on the FULL training dataset
print("Training the final model on the full training dataset (this may take a minute)...")
best_rf.fit(x_train, y_train)
print("Final model training complete.")


Training the final model on the full training dataset (this may take a minute)...
Final model training complete.


# Evaluate Final Model on Test Set

In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

y_pred = best_rf.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

print("--- Overall Metrics ---")
print(f"Accuracy: {accuracy:.4f}")

print("\n--- Class 1 (Minority) Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")

print("\n--- Class 0 (Majority) Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred, pos_label=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, pos_label=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred, pos_label=0):.4f}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))


--- Overall Metrics ---
Accuracy: 0.8253

--- Class 1 (Minority) Metrics ---
Precision: 0.3084
Recall:    0.4120
F1-score:  0.3528

--- Class 0 (Majority) Metrics ---
Precision: 0.9197
Recall:    0.8793
F1-score:  0.8990

--- Confusion Matrix ---
[[39719  5451]
 [ 3469  2431]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.88      0.90     45170
           1       0.31      0.41      0.35      5900

    accuracy                           0.83     51070
   macro avg       0.61      0.65      0.63     51070
weighted avg       0.85      0.83      0.84     51070



In [10]:
# Save Random Forest model
joblib.dump(best_rf, "random_forestfinal_model.pkl")

['random_forestfinal_model.pkl']